# Chapter 3 — Linear Regression (ISLR)
## Advertising Dataset

Follows Sections 3.1–3.2 of *An Introduction to Statistical Learning*.  
Dataset: `advertising_budget_and_sales.csv` — 200 markets, 3 ad channels, 1 sales target.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from mpl_toolkits.mplot3d import Axes3D
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats
import os

# Output directory for figures
os.makedirs("../output", exist_ok=True)

df = pd.read_csv("../data/advertising_budget_and_sales.csv", index_col=0)
df.columns = ["TV", "Radio", "Newspaper", "Sales"]
df.head()

ModuleNotFoundError: No module named 'numpy'

## 3.1 Simple Linear Regression

### 3.1.1 Estimating the Coefficients

We fit the model:

$$\hat{Y} = \hat{\beta}_0 + \hat{\beta}_1 X$$

The least squares estimates minimise the **Residual Sum of Squares (RSS)**:

$$\text{RSS} = \sum_{i=1}^{n}(y_i - \hat{\beta}_0 - \hat{\beta}_1 x_i)^2$$

Closed-form solution:

$$\hat{\beta}_1 = \frac{\sum(x_i - \bar{x})(y_i - \bar{y})}{\sum(x_i - \bar{x})^2}, \qquad \hat{\beta}_0 = \bar{y} - \hat{\beta}_1\bar{x}$$

In [ ]:
# --- Simple linear regression: Sales ~ TV ---
model_tv = smf.ols("Sales ~ TV", data=df).fit()
b0, b1 = model_tv.params["Intercept"], model_tv.params["TV"]
print(f"β̂₀ = {b0:.4f},  β̂₁ = {b1:.4f}")

# Predictions and residuals
x = df["TV"].values
y = df["Sales"].values
y_hat = model_tv.fittedvalues.values

# --- Figure 3.1: Least squares fit with residuals ---
fig, ax = plt.subplots(figsize=(7, 5))

for xi, yi, yhi in zip(x, y, y_hat):
    ax.plot([xi, xi], [yi, yhi], color="grey", linewidth=0.7, zorder=1)

ax.scatter(x, y, color="red", s=15, zorder=2)
x_line = np.linspace(x.min(), x.max(), 200)
ax.plot(x_line, b0 + b1 * x_line, color="steelblue", linewidth=2, zorder=3)

ax.set_xlabel("TV")
ax.set_ylabel("Sales")
ax.set_title("Figure 3.1 — Least squares fit: Sales ~ TV\n(grey segments = residuals)")
plt.tight_layout()
plt.savefig("../output/fig3_1_slr_tv.png", dpi=150)
plt.show()

In [ ]:
# --- Figure 3.2: RSS surface over a grid of (β₀, β₁) values ---
b0_grid = np.linspace(5, 9, 100)
b1_grid = np.linspace(0.03, 0.07, 100)
B0, B1 = np.meshgrid(b0_grid, b1_grid)

RSS = np.array([
    np.sum((y - b0v - b1v * x) ** 2)
    for b0v, b1v in zip(B0.ravel(), B1.ravel())
]).reshape(B0.shape)

fig = plt.figure(figsize=(12, 5))

# Contour plot
ax1 = fig.add_subplot(1, 2, 1)
levels = [2.15, 2.2, 2.3, 2.5, 3.0]
cs = ax1.contour(B0, B1, RSS, levels=levels, colors="black", linewidths=0.8)
ax1.clabel(cs, fmt="%.2f", fontsize=8)
ax1.scatter([b0], [b1], color="red", zorder=5, s=50, label=f"OLS ({b0:.2f}, {b1:.4f})")
ax1.set_xlabel("β₀")
ax1.set_ylabel("β₁")
ax1.set_title("Figure 3.2 (left) — RSS contours")
ax1.legend(fontsize=8)

# 3-D surface
ax2 = fig.add_subplot(1, 2, 2, projection="3d")
ax2.plot_surface(B0, B1, RSS, cmap="viridis", alpha=0.8, linewidth=0)
ax2.scatter([b0], [b1], [model_tv.ssr], color="red", s=60, zorder=5)
ax2.set_xlabel("β₀")
ax2.set_ylabel("β₁")
ax2.set_zlabel("RSS")
ax2.set_title("Figure 3.2 (right) — RSS surface")

plt.tight_layout()
plt.savefig("../output/fig3_2_rss_surface.png", dpi=150)
plt.show()

### 3.1.2 Assessing the Accuracy of the Coefficient Estimates

The **standard error** of $\hat{\beta}_1$ and $\hat{\beta}_0$ are:

$$SE(\hat{\beta}_1)^2 = \frac{\sigma^2}{\sum(x_i - \bar{x})^2}, \qquad SE(\hat{\beta}_0)^2 = \sigma^2\left[\frac{1}{n} + \frac{\bar{x}^2}{\sum(x_i-\bar{x})^2}\right]$$

A **95% confidence interval** for $\beta_1$: $\hat{\beta}_1 \pm 2 \cdot SE(\hat{\beta}_1)$

The **t-statistic**: $t = \hat{\beta}_1 / SE(\hat{\beta}_1)$ tests $H_0: \beta_1 = 0$

#### Table 3.1 — Coefficient estimates (Sales ~ TV)

In [ ]:
# Table 3.1
table31 = pd.DataFrame({
    "Coefficient": model_tv.params,
    "Std. error": model_tv.bse,
    "t-statistic": model_tv.tvalues,
    "p-value": model_tv.pvalues,
})
table31.index = ["Intercept", "TV"]
table31.round(4)

In [ ]:
# 95% Confidence intervals for β₀ and β₁
ci = model_tv.conf_int(alpha=0.05)
ci.index = ["Intercept", "TV"]
ci.columns = ["Lower 95%", "Upper 95%"]
print("95% Confidence Intervals:")
print(ci.round(3))

In [ ]:
# --- Figure 3.3: Population vs least squares regression line (simulated) ---
np.random.seed(42)
true_b0, true_b1, n_sim = 2, 3, 100
X_sim = np.random.uniform(-2, 2, n_sim)
Y_sim = true_b0 + true_b1 * X_sim + np.random.normal(0, 1, n_sim)
ols_b0, ols_b1, *_ = stats.linregress(X_sim, Y_sim)

x_range = np.linspace(-2, 2, 200)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Left: one dataset
ax1.scatter(X_sim, Y_sim, color="black", s=10)
ax1.plot(x_range, true_b0 + true_b1 * x_range, color="red", label="Population: Y = 2 + 3X")
ax1.plot(x_range, ols_b0 + ols_b1 * x_range, color="steelblue", label="Least squares fit")
ax1.set_xlabel("X"); ax1.set_ylabel("Y")
ax1.set_title("Figure 3.3 (left) — One sample")
ax1.legend(fontsize=8)

# Right: ten datasets
ax2.plot(x_range, true_b0 + true_b1 * x_range, color="red", linewidth=2, label="Population")
for _ in range(10):
    Yi = true_b0 + true_b1 * X_sim + np.random.normal(0, 1, n_sim)
    b0i, b1i, *_ = stats.linregress(X_sim, Yi)
    ax2.plot(x_range, b0i + b1i * x_range, color="lightblue", linewidth=0.9)
ax2.plot(x_range, ols_b0 + ols_b1 * x_range, color="darkblue", linewidth=1.5, label="OLS (first sample)")
ax2.set_xlabel("X"); ax2.set_ylabel("Y")
ax2.set_title("Figure 3.3 (right) — Ten samples")
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig("../output/fig3_3_population_vs_ols.png", dpi=150)
plt.show()

### 3.1.3 Assessing the Accuracy of the Model

Two key metrics:

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **RSE** | $\sqrt{\frac{RSS}{n-2}}$ | Avg. deviation from regression line (same units as Y) |
| **R²** | $1 - \frac{RSS}{TSS}$ | Proportion of variance in Y explained by X (0–1) |

#### Table 3.2 — Model accuracy (Sales ~ TV)

In [ ]:
# Table 3.2
rse = np.sqrt(model_tv.ssr / (len(df) - 2))
r2  = model_tv.rsquared
f_stat = model_tv.fvalue

table32 = pd.DataFrame({
    "Value": [round(rse, 2), round(r2, 3), round(f_stat, 1)]
}, index=["RSE", "R²", "F-statistic"])
table32

## 3.2 Multiple Linear Regression

The model extends to $p$ predictors:

$$Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \cdots + \beta_p X_p + \epsilon$$

For the Advertising data:

$$\text{Sales} = \beta_0 + \beta_1 \times \text{TV} + \beta_2 \times \text{Radio} + \beta_3 \times \text{Newspaper} + \epsilon$$

### 3.2.1 Simple Regressions on Radio and Newspaper (Table 3.3)

In [ ]:
def coef_table(model, predictor_label):
    params = model.params
    bse   = model.bse
    tvals = model.tvalues
    pvals = model.pvalues
    return pd.DataFrame({
        "Coefficient": params.values,
        "Std. error":  bse.values,
        "t-statistic": tvals.values,
        "p-value":     pvals.values,
    }, index=["Intercept", predictor_label])

model_radio = smf.ols("Sales ~ Radio", data=df).fit()
model_news  = smf.ols("Sales ~ Newspaper", data=df).fit()

print("Sales ~ Radio")
display(coef_table(model_radio, "Radio").round(3))

print("\nSales ~ Newspaper")
display(coef_table(model_news, "Newspaper").round(3))

### 3.2.1 Multiple Regression Coefficient Estimates (Table 3.4)

Note: the `Newspaper` coefficient becomes near-zero and insignificant once TV and Radio are included — newspaper is a **surrogate** for radio (correlated).

In [ ]:
# Table 3.4 — Multiple linear regression
model_mlr = smf.ols("Sales ~ TV + Radio + Newspaper", data=df).fit()

table34 = pd.DataFrame({
    "Coefficient": model_mlr.params,
    "Std. error":  model_mlr.bse,
    "t-statistic": model_mlr.tvalues,
    "p-value":     model_mlr.pvalues,
})
table34.index = ["Intercept", "TV", "Radio", "Newspaper"]
table34.round(4)

In [ ]:
# Table 3.5 — Correlation matrix
table35 = df[["TV", "Radio", "Newspaper", "Sales"]].corr()
table35.round(4)

### 3.2.2 Some Important Questions

**Question 1:** Is at least one predictor useful? → **F-statistic**

$$F = \frac{(TSS - RSS)/p}{RSS/(n-p-1)}$$

If $H_0$ (all $\beta_j = 0$) is true, $F \approx 1$. A large $F$ rejects $H_0$.

#### Table 3.6 — MLR model accuracy

In [ ]:
# Table 3.6
rse_mlr = np.sqrt(model_mlr.ssr / (len(df) - 4))  # n - p - 1 = 200 - 3 - 1

table36 = pd.DataFrame({
    "Value": [round(rse_mlr, 2), round(model_mlr.rsquared, 3), round(model_mlr.fvalue, 1)]
}, index=["RSE", "R²", "F-statistic"])
table36

In [ ]:
# --- Figure 3.4: Least squares plane (Sales ~ TV + Radio, ignoring Newspaper) ---
from matplotlib import cm

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")

tv_grid   = np.linspace(df["TV"].min(),    df["TV"].max(),    30)
radio_grid = np.linspace(df["Radio"].min(), df["Radio"].max(), 30)
TV_g, R_g = np.meshgrid(tv_grid, radio_grid)

model_tv_radio = smf.ols("Sales ~ TV + Radio", data=df).fit()
p = model_tv_radio.params
Z = p["Intercept"] + p["TV"] * TV_g + p["Radio"] * R_g

ax.plot_surface(TV_g, R_g, Z, alpha=0.5, cmap=cm.Blues)
ax.scatter(df["TV"], df["Radio"], df["Sales"], color="red", s=8)

ax.set_xlabel("TV"); ax.set_ylabel("Radio"); ax.set_zlabel("Sales")
ax.set_title("Figure 3.4 — Least squares plane (TV + Radio)")

plt.tight_layout()
plt.savefig("../output/fig3_4_mlr_plane.png", dpi=150)
plt.show()

In [ ]:
# --- Summary: SLR vs MLR comparison ---
summary = pd.DataFrame({
    "Model":       ["Sales ~ TV", "Sales ~ Radio", "Sales ~ Newspaper", "Sales ~ TV + Radio + Newspaper"],
    "R²":          [model_tv.rsquared, model_radio.rsquared, model_news.rsquared, model_mlr.rsquared],
    "RSE":         [np.sqrt(m.ssr / (len(df)-2)) for m in [model_tv, model_radio, model_news]]
             + [np.sqrt(model_mlr.ssr / (len(df)-4))],
    "F-statistic": [model_tv.fvalue, model_radio.fvalue, model_news.fvalue, model_mlr.fvalue],
}).set_index("Model")

summary.round(3)